In [0]:
# ============================================================================
# PRICING FEATURE ADOPTION AUDIT - COMPLETE ANALYSIS
# Hierarchy: GMV Tier → Supplier Segment → Activity Type → Tour Features
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print("="*100)
print("PRICING FEATURE ADOPTION AUDIT")
print("="*100)
print(f"Analysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d')}")
print("="*100)


In [0]:
# ============================================================================
# STEP 1: LOAD DATA & CREATE SUPPLIER GMV TIERS
# ============================================================================

print("\n[1/6] Loading combined data...")

# Load tour-level data
df_spark = spark.table("production.supply_analytics.pricing_feature_combined")

# Create supplier-level GMV aggregation and assign tiers
supplier_gmv = df_spark.filter(F.col("gmv_l12mo") > 0) \
    .groupBy("supplier_id") \
    .agg(
        F.sum("gmv_l12mo").alias("supplier_gmv_l12mo"),
        F.sum("bookings_l12mo").alias("supplier_bookings_l12mo"),
        F.count("tour_id").alias("supplier_tour_count")
    )

# Calculate GMV quartiles for supplier tiers
window_spec = Window.orderBy("supplier_gmv_l12mo")
supplier_gmv = supplier_gmv.withColumn(
    "gmv_quartile", 
    F.ntile(4).over(window_spec)
).withColumn(
    "gmv_tier",
    F.when(F.col("gmv_quartile") == 1, F.lit("Bottom 25% (Emerging)"))
     .when(F.col("gmv_quartile") == 2, F.lit("25-50% (Growing)"))
     .when(F.col("gmv_quartile") == 3, F.lit("50-75% (Established)"))
     .when(F.col("gmv_quartile") == 4, F.lit("Top 25% (Top Performers)"))
     .otherwise(F.lit("Unknown"))
)

# Select only needed columns from supplier_gmv
supplier_tier_mapping = supplier_gmv.select(
    "supplier_id", 
    F.col("gmv_tier").alias("supplier_gmv_tier"),
    F.col("supplier_gmv_l12mo").alias("total_supplier_gmv_l12mo"),
    "supplier_tour_count"
)

# Join GMV tier back to tour-level data
df_spark = df_spark.join(
    supplier_tier_mapping,
    on="supplier_id",
    how="left"
)

# Convert to Pandas for analysis
df = df_spark.toPandas()

print(f"\nData Loaded:")
print(f"  - Total Tours: {len(df):,}")
print(f"  - Total Suppliers: {df['supplier_id'].nunique():,}")

# ============================================================================
# CONVERT DECIMAL COLUMNS TO FLOAT
# ============================================================================
decimal_columns = [
    'gmv_l3mo', 'gmv_l6mo', 'gmv_l12mo', 'gmv_lifetime',
    'nr_l3mo', 'nr_l6mo', 'nr_l12mo', 'nr_lifetime',
    'avg_booking_value_l12mo', 'avg_tickets_per_booking_l12mo',
    'total_supplier_gmv_l12mo', 'repeat_customer_rate_l3mo',
    'repeat_customer_rate_l6mo', 'repeat_customer_rate_l12mo',
    'gmv_percentile', 'bookings_percentile', 'nr_percentile', 'basket_size_percentile'
]

for col in decimal_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"  - Total GMV (L12M): €{df['gmv_l12mo'].sum():,.0f}")
print(f"  - Tours with GMV > 0: {len(df[df['gmv_l12mo'] > 0]):,}")

# Filter to active tours only
df = df[df['gmv_l12mo'] > 0].copy()

# Rename for consistency
df['gmv_tier'] = df['supplier_gmv_tier']

print(f"\nAfter filtering to active tours (GMV > 0):")
print(f"  - Tours: {len(df):,}")
print(f"  - Suppliers: {df['supplier_id'].nunique():,}")
print(f"\nGMV Tier Distribution:")
for tier in df['gmv_tier'].value_counts().sort_index().items():
    print(f"  - {tier[0]}: {tier[1]:,} tours")


In [0]:
# ============================================================================
# STEP 2: DATA PREPARATION
# ============================================================================

print("\n[2/6] Preparing data and creating segments...")

# Fill missing values
df['supplier_segment'] = df['supplier_segment'].fillna('Unknown')
df['activity_type'] = df['activity_type'].fillna('Unknown')
df['gmv_tier'] = df['gmv_tier'].fillna('Unknown')

# Define pricing features
PRICING_FEATURES = {
    'has_group_pricing': 'Group Pricing',
    'has_addons': 'Add-ons',
    'has_scale_pricing': 'Scale Pricing',
    'uses_live_pricing': 'Live Pricing',
    'uses_api_pricing': 'API Pricing'
}

# Define tier order with percentage labels
GMV_TIERS = ['Bottom 25% (Emerging)', '25-50% (Growing)', '50-75% (Established)', 'Top 25% (Top Performers)']

# Get top supplier segments and activity types
top_segments = df['supplier_segment'].value_counts().head(10).index.tolist()
top_activities = df['activity_type'].value_counts().head(15).index.tolist()

print(f"\nSegmentation Summary:")
print(f"  - GMV Tiers:")
gmv_tier_counts = df['gmv_tier'].value_counts()
for tier in GMV_TIERS:
    if tier in gmv_tier_counts.index:
        count = gmv_tier_counts[tier]
        pct = 100 * count / len(df)
        print(f"      {tier}: {count:,} tours ({pct:.1f}%)")

print(f"\nTop 5 Supplier Segments:")
for seg, count in df['supplier_segment'].value_counts().head(5).items():
    print(f"    {seg}: {count:,} tours")
    
print(f"\nTop 5 Activity Types:")
for act, count in df['activity_type'].value_counts().head(5).items():
    print(f"    {act}: {count:,} tours")


In [0]:
# ============================================================================
# STEP 3: EXECUTIVE SUMMARY - OVERALL ADOPTION
# ============================================================================

print("\n[3/6] Creating executive summary...")

fig, axes = plt.subplots(2, 3, figsize=(22, 12))
fig.suptitle('PRICING FEATURE ADOPTION - EXECUTIVE SUMMARY', fontsize=20, fontweight='bold', y=0.995)

# Chart 1: Overall Feature Adoption Rates
ax1 = axes[0, 0]
overall_adoption = {}
for col, name in PRICING_FEATURES.items():
    overall_adoption[name] = 100 * df[col].sum() / len(df)

colors = ['#2ca02c', '#ff7f0e', '#9467bd', '#17becf', '#d62728']
bars = ax1.barh(list(overall_adoption.keys()), list(overall_adoption.values()), color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax1.set_xlabel('Adoption Rate (%)', fontweight='bold', fontsize=12)
ax1.set_title('Overall Feature Adoption Rates', fontweight='bold', fontsize=14, pad=15)
ax1.grid(axis='x', alpha=0.3)

for i, (feature, rate) in enumerate(overall_adoption.items()):
    tours_using = df[list(PRICING_FEATURES.keys())[i]].sum()
    ax1.text(rate + 1, i, f'{rate:.1f}% ({tours_using:,} tours)', va='center', fontsize=10, fontweight='bold')

# Chart 2: GMV Tier Distribution
ax2 = axes[0, 1]
gmv_dist = df.groupby('gmv_tier').agg({'tour_id': 'count', 'gmv_l12mo': 'sum'}).reindex(GMV_TIERS)
colors_tier = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']
bars = ax2.bar(range(len(gmv_dist)), gmv_dist['tour_id'], color=colors_tier, alpha=0.8, edgecolor='black', linewidth=1.5)
ax2.set_xticks(range(len(gmv_dist)))
ax2.set_xticklabels(gmv_dist.index, fontsize=10, fontweight='bold')
ax2.set_ylabel('Number of Tours', fontweight='bold', fontsize=12)
ax2.set_title('Tours by GMV Tier', fontweight='bold', fontsize=14, pad=15)
ax2.grid(axis='y', alpha=0.3)

for i, (tours, gmv) in enumerate(zip(gmv_dist['tour_id'], gmv_dist['gmv_l12mo'])):
    ax2.text(i, tours + tours*0.02, f'{tours:,}\n€{gmv/1e6:.1f}M', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Chart 3: Supplier Segment Distribution
ax3 = axes[0, 2]
segment_dist = df['supplier_segment'].value_counts().head(8).sort_values(ascending=True)
bars = ax3.barh(range(len(segment_dist)), segment_dist.values, color='#9467bd', alpha=0.8, edgecolor='black', linewidth=1.5)
ax3.set_yticks(range(len(segment_dist)))
ax3.set_yticklabels(segment_dist.index, fontsize=9)
ax3.set_xlabel('Number of Tours', fontweight='bold', fontsize=12)
ax3.set_title('Top Supplier Segments', fontweight='bold', fontsize=14, pad=15)
ax3.grid(axis='x', alpha=0.3)

for i, count in enumerate(segment_dist.values):
    ax3.text(count + count*0.02, i, f'{count:,}', va='center', fontsize=9, fontweight='bold')

# Chart 4: Activity Type Distribution
ax4 = axes[1, 0]
activity_dist = df['activity_type'].value_counts().head(10).sort_values(ascending=True)
bars = ax4.barh(range(len(activity_dist)), activity_dist.values, color='#17becf', alpha=0.8, edgecolor='black', linewidth=1.5)
ax4.set_yticks(range(len(activity_dist)))
ax4.set_yticklabels(activity_dist.index, fontsize=9)
ax4.set_xlabel('Number of Tours', fontweight='bold', fontsize=12)
ax4.set_title('Top 10 Activity Types', fontweight='bold', fontsize=14, pad=15)
ax4.grid(axis='x', alpha=0.3)

for i, count in enumerate(activity_dist.values):
    ax4.text(count + count*0.02, i, f'{count:,}', va='center', fontsize=9, fontweight='bold')

# Chart 5: Feature Adoption by GMV Tier (Heatmap)
ax5 = axes[1, 1]
heatmap_data = []
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    row = []
    for col in PRICING_FEATURES.keys():
        adoption = 100 * tier_df[col].sum() / len(tier_df) if len(tier_df) > 0 else 0
        row.append(adoption)
    heatmap_data.append(row)

heatmap_df = pd.DataFrame(heatmap_data, columns=list(PRICING_FEATURES.values()), index=GMV_TIERS)
sns.heatmap(heatmap_df, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax5, cbar_kws={'label': 'Adoption %'}, 
            linewidths=1, linecolor='white', vmin=0, vmax=50)
ax5.set_xlabel('Pricing Features', fontweight='bold', fontsize=12)
ax5.set_ylabel('GMV Tier', fontweight='bold', fontsize=12)
ax5.set_title('Adoption % by GMV Tier', fontweight='bold', fontsize=14, pad=15)
ax5.set_yticklabels(GMV_TIERS, rotation=0, fontsize=10)

# Chart 6: Multi-Feature Adoption
ax6 = axes[1, 2]
df['num_features'] = df[[col for col in PRICING_FEATURES.keys()]].sum(axis=1)
feature_count_dist = df['num_features'].value_counts().sort_index()
colors_features = ['#d62728', '#ff7f0e', '#bcbd22', '#2ca02c', '#1f77b4', '#9467bd']
bars = ax6.bar(range(len(feature_count_dist)), feature_count_dist.values, 
               color=colors_features[:len(feature_count_dist)], alpha=0.8, edgecolor='black', linewidth=1.5)
ax6.set_xticks(range(len(feature_count_dist)))
ax6.set_xticklabels([f'{int(x)} features' for x in feature_count_dist.index], fontsize=10)
ax6.set_ylabel('Number of Tours', fontweight='bold', fontsize=12)
ax6.set_title('Tours by Feature Count', fontweight='bold', fontsize=14, pad=15)
ax6.grid(axis='y', alpha=0.3)

for i, count in enumerate(feature_count_dist.values):
    pct = 100 * count / len(df)
    ax6.text(i, count + count*0.02, f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Executive Summary created")


In [0]:
# ============================================================================
# STEP 4: DEEP DIVE BY GMV TIER
# ============================================================================

print("\n[4/6] Creating GMV tier deep dives...")

# Updated tier colors to match new tier names
tier_colors = {
    'Bottom 25% (Emerging)': '#d62728', 
    '25-50% (Growing)': '#ff7f0e', 
    '50-75% (Established)': '#2ca02c', 
    'Top 25% (Top Performers)': '#1f77b4'
}

for tier_idx, tier in enumerate(GMV_TIERS):
    tier_df = df[df['gmv_tier'] == tier].copy()
    
    if len(tier_df) == 0:
        continue
    
    print(f"\nAnalyzing {tier} tier: {len(tier_df):,} tours, {tier_df['supplier_id'].nunique():,} suppliers")
    
    fig = plt.figure(figsize=(24, 16))
    gs = fig.add_gridspec(4, 3, hspace=0.35, wspace=0.3, top=0.95)
    
    tier_color = tier_colors[tier]
    
    fig.suptitle(f'{tier.upper()} - PRICING FEATURE ADOPTION ANALYSIS\n{len(tier_df):,} tours | {tier_df["supplier_id"].nunique():,} suppliers | €{tier_df["gmv_l12mo"].sum()/1e6:.1f}M GMV', 
                 fontsize=18, fontweight='bold', y=0.98)
    
    # ========== ROW 1: OVERVIEW ==========
    
    # Chart 1.1: Feature Adoption in this Tier
    ax1 = fig.add_subplot(gs[0, 0])
    tier_adoption = {}
    for col, name in PRICING_FEATURES.items():
        tier_adoption[name] = 100 * tier_df[col].sum() / len(tier_df)
    
    colors_feat = ['#2ca02c', '#ff7f0e', '#9467bd', '#17becf', '#d62728']
    bars = ax1.barh(list(tier_adoption.keys()), list(tier_adoption.values()), 
                    color=colors_feat, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax1.set_xlabel('Adoption Rate (%)', fontweight='bold', fontsize=11)
    ax1.set_title(f'Feature Adoption Rates', fontweight='bold', fontsize=13, pad=10)
    ax1.grid(axis='x', alpha=0.3)
    
    for i, (feature, rate) in enumerate(tier_adoption.items()):
        tours = tier_df[list(PRICING_FEATURES.keys())[i]].sum()
        ax1.text(rate + 1, i, f'{rate:.1f}% ({tours:,})', va='center', fontsize=9, fontweight='bold')
    
    # Chart 1.2: Supplier Segment Breakdown
    ax2 = fig.add_subplot(gs[0, 1])
    segment_breakdown = tier_df['supplier_segment'].value_counts().head(8).sort_values(ascending=True)
    bars = ax2.barh(range(len(segment_breakdown)), segment_breakdown.values, 
                    color=tier_color, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax2.set_yticks(range(len(segment_breakdown)))
    ax2.set_yticklabels(segment_breakdown.index, fontsize=9)
    ax2.set_xlabel('Number of Tours', fontweight='bold', fontsize=11)
    ax2.set_title(f'Top Supplier Segments', fontweight='bold', fontsize=13, pad=10)
    ax2.grid(axis='x', alpha=0.3)
    
    for i, count in enumerate(segment_breakdown.values):
        pct = 100 * count / len(tier_df)
        ax2.text(count + count*0.02, i, f'{count:,} ({pct:.1f}%)', va='center', fontsize=8, fontweight='bold')
    
    # Chart 1.3: Activity Type Breakdown
    ax3 = fig.add_subplot(gs[0, 2])
    activity_breakdown = tier_df['activity_type'].value_counts().head(10).sort_values(ascending=True)
    bars = ax3.barh(range(len(activity_breakdown)), activity_breakdown.values, 
                    color=tier_color, alpha=0.6, edgecolor='black', linewidth=1.5)
    ax3.set_yticks(range(len(activity_breakdown)))
    ax3.set_yticklabels(activity_breakdown.index, fontsize=8)
    ax3.set_xlabel('Number of Tours', fontweight='bold', fontsize=11)
    ax3.set_title(f'Top Activity Types', fontweight='bold', fontsize=13, pad=10)
    ax3.grid(axis='x', alpha=0.3)
    
    for i, count in enumerate(activity_breakdown.values):
        pct = 100 * count / len(tier_df)
        ax3.text(count + count*0.02, i, f'{count:,} ({pct:.1f}%)', va='center', fontsize=8, fontweight='bold')
    
    # ========== ROW 2: BY SUPPLIER SEGMENT ==========
    
    # Chart 2.1: Feature Adoption by Supplier Segment (Heatmap)
    ax4 = fig.add_subplot(gs[1, :])
    top_segs_tier = tier_df['supplier_segment'].value_counts().head(8).index.tolist()
    heatmap_seg = []
    for seg in top_segs_tier:
        seg_df = tier_df[tier_df['supplier_segment'] == seg]
        row = []
        for col in PRICING_FEATURES.keys():
            adoption = 100 * seg_df[col].sum() / len(seg_df) if len(seg_df) > 0 else 0
            row.append(adoption)
        heatmap_seg.append(row)
    
    heatmap_seg_df = pd.DataFrame(heatmap_seg, columns=list(PRICING_FEATURES.values()), index=top_segs_tier)
    sns.heatmap(heatmap_seg_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax4, 
                cbar_kws={'label': 'Adoption %'}, linewidths=1.5, linecolor='white', vmin=0, vmax=60)
    ax4.set_xlabel('Pricing Features', fontweight='bold', fontsize=12)
    ax4.set_ylabel('Supplier Segment', fontweight='bold', fontsize=12)
    ax4.set_title(f'Feature Adoption by Supplier Segment (% of tours)', fontweight='bold', fontsize=14, pad=15)
    ax4.set_yticklabels(top_segs_tier, rotation=0, fontsize=10)
    
    # ========== ROW 3: BY ACTIVITY TYPE ==========
    
    # Chart 3.1: Feature Adoption by Activity Type (Heatmap)
    ax5 = fig.add_subplot(gs[2, :])
    top_acts_tier = tier_df['activity_type'].value_counts().head(10).index.tolist()
    heatmap_act = []
    for act in top_acts_tier:
        act_df = tier_df[tier_df['activity_type'] == act]
        row = []
        for col in PRICING_FEATURES.keys():
            adoption = 100 * act_df[col].sum() / len(act_df) if len(act_df) > 0 else 0
            row.append(adoption)
        heatmap_act.append(row)
    
    heatmap_act_df = pd.DataFrame(heatmap_act, columns=list(PRICING_FEATURES.values()), index=top_acts_tier)
    sns.heatmap(heatmap_act_df, annot=True, fmt='.1f', cmap='Blues', ax=ax5, 
                cbar_kws={'label': 'Adoption %'}, linewidths=1.5, linecolor='white', vmin=0, vmax=60)
    ax5.set_xlabel('Pricing Features', fontweight='bold', fontsize=12)
    ax5.set_ylabel('Activity Type', fontweight='bold', fontsize=12)
    ax5.set_title(f'Feature Adoption by Activity Type (% of tours)', fontweight='bold', fontsize=14, pad=15)
    ax5.set_yticklabels(top_acts_tier, rotation=0, fontsize=9)
    
    # ========== ROW 4: CROSS-SEGMENT ANALYSIS ==========
    
    # Chart 4.1: Tours by Segment × Activity (Top combinations)
    ax6 = fig.add_subplot(gs[3, 0])
    seg_act_combo = tier_df.groupby(['supplier_segment', 'activity_type']).size().reset_index(name='tours')
    seg_act_combo = seg_act_combo.sort_values('tours', ascending=False).head(10)
    seg_act_combo['label'] = seg_act_combo['supplier_segment'] + '\n' + seg_act_combo['activity_type']
    
    bars = ax6.barh(range(len(seg_act_combo)), seg_act_combo['tours'].values, 
                    color=tier_color, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax6.set_yticks(range(len(seg_act_combo)))
    ax6.set_yticklabels(seg_act_combo['label'].values, fontsize=7)
    ax6.set_xlabel('Number of Tours', fontweight='bold', fontsize=11)
    ax6.set_title(f'Top 10 Segment × Activity Combinations', fontweight='bold', fontsize=13, pad=10)
    ax6.grid(axis='x', alpha=0.3)
    
    for i, count in enumerate(seg_act_combo['tours'].values):
        ax6.text(count + count*0.02, i, f'{count:,}', va='center', fontsize=8, fontweight='bold')
    
    # Chart 4.2: Multi-Feature Adoption Distribution
    ax7 = fig.add_subplot(gs[3, 1])
    feature_count = tier_df['num_features'].value_counts().sort_index()
    colors_multi = ['#d62728', '#ff7f0e', '#bcbd22', '#2ca02c', '#1f77b4', '#9467bd']
    bars = ax7.bar(range(len(feature_count)), feature_count.values, 
                   color=colors_multi[:len(feature_count)], alpha=0.8, edgecolor='black', linewidth=1.5)
    ax7.set_xticks(range(len(feature_count)))
    ax7.set_xticklabels([f'{int(x)}' for x in feature_count.index], fontsize=10)
    ax7.set_xlabel('Number of Features Used', fontweight='bold', fontsize=11)
    ax7.set_ylabel('Number of Tours', fontweight='bold', fontsize=11)
    ax7.set_title(f'Feature Sophistication', fontweight='bold', fontsize=13, pad=10)
    ax7.grid(axis='y', alpha=0.3)
    
    for i, count in enumerate(feature_count.values):
        pct = 100 * count / len(tier_df)
        ax7.text(i, count + count*0.02, f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    # Chart 4.3: Summary Stats Table
    ax8 = fig.add_subplot(gs[3, 2])
    ax8.axis('off')
    
    summary_stats = [
        ['Total Tours', f'{len(tier_df):,}'],
        ['Total Suppliers', f'{tier_df["supplier_id"].nunique():,}'],
        ['Total GMV', f'€{tier_df["gmv_l12mo"].sum()/1e6:.1f}M'],
        ['Avg Tours/Supplier', f'{len(tier_df)/tier_df["supplier_id"].nunique():.1f}'],
        ['Avg GMV/Tour', f'€{tier_df["gmv_l12mo"].mean():,.0f}'],
        ['Supplier Segments', f'{tier_df["supplier_segment"].nunique():,}'],
        ['Activity Types', f'{tier_df["activity_type"].nunique():,}'],
        ['Tours with 0 features', f'{len(tier_df[tier_df["num_features"]==0]):,} ({100*len(tier_df[tier_df["num_features"]==0])/len(tier_df):.1f}%)'],
        ['Tours with 1+ features', f'{len(tier_df[tier_df["num_features"]>=1]):,} ({100*len(tier_df[tier_df["num_features"]>=1])/len(tier_df):.1f}%)'],
        ['Tours with 3+ features', f'{len(tier_df[tier_df["num_features"]>=3]):,} ({100*len(tier_df[tier_df["num_features"]>=3])/len(tier_df):.1f}%)']
    ]
    
    table = ax8.table(cellText=summary_stats, cellLoc='left', loc='center', 
                     colWidths=[0.6, 0.4], bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.5)
    
    for i in range(len(summary_stats)):
        table[(i, 0)].set_facecolor('#f0f0f0')
        table[(i, 0)].set_text_props(weight='bold')
        table[(i, 1)].set_facecolor('white')
    
    ax8.set_title(f'Tier Summary', fontweight='bold', fontsize=14, pad=20, y=0.98)
    
    plt.tight_layout()
    plt.show()
    
    print(f"✓ {tier} tier analysis complete")


In [0]:
# ============================================================================
# STEP 5: GRANULAR ANALYSIS - SEGMENT × ACTIVITY WITHIN GMV TIERS
# ============================================================================

print("\n[5/6] Creating granular segment × activity analysis...")

for tier_idx, tier in enumerate(GMV_TIERS):
    tier_df = df[df['gmv_tier'] == tier].copy()
    
    if len(tier_df) == 0:
        continue
    
    # Get top 5 segments for this tier
    top_5_segments = tier_df['supplier_segment'].value_counts().head(5).index.tolist()
    
    for seg_idx, segment in enumerate(top_5_segments):
        seg_df = tier_df[tier_df['supplier_segment'] == segment].copy()
        
        if len(seg_df) < 10:  # Skip if too few tours
            continue
        
        print(f"\n  Analyzing {tier} → {segment}: {len(seg_df):,} tours")
        
        fig, axes = plt.subplots(2, 3, figsize=(22, 12))
        fig.suptitle(f'{tier.upper()} → {segment.upper()}\n{len(seg_df):,} tours | {seg_df["supplier_id"].nunique():,} suppliers | €{seg_df["gmv_l12mo"].sum()/1e6:.2f}M GMV', 
                     fontsize=16, fontweight='bold', y=0.98)
        
        # Chart 1: Feature Adoption in this Segment
        ax1 = axes[0, 0]
        seg_adoption = {}
        for col, name in PRICING_FEATURES.items():
            seg_adoption[name] = 100 * seg_df[col].sum() / len(seg_df)
        
        colors_feat = ['#2ca02c', '#ff7f0e', '#9467bd', '#17becf', '#d62728']
        bars = ax1.barh(list(seg_adoption.keys()), list(seg_adoption.values()), 
                        color=colors_feat, alpha=0.8, edgecolor='black', linewidth=1.5)
        ax1.set_xlabel('Adoption Rate (%)', fontweight='bold')
        ax1.set_title(f'Feature Adoption', fontweight='bold', pad=10)
        ax1.grid(axis='x', alpha=0.3)
        
        for i, (feature, rate) in enumerate(seg_adoption.items()):
            tours = seg_df[list(PRICING_FEATURES.keys())[i]].sum()
            ax1.text(rate + 1, i, f'{rate:.1f}% ({tours:,})', va='center', fontsize=9, fontweight='bold')
        
        # Chart 2: Activity Type Distribution
        ax2 = axes[0, 1]
        act_dist = seg_df['activity_type'].value_counts().head(8).sort_values(ascending=True)
        bars = ax2.barh(range(len(act_dist)), act_dist.values, color='#17becf', alpha=0.8, edgecolor='black', linewidth=1.5)
        ax2.set_yticks(range(len(act_dist)))
        ax2.set_yticklabels(act_dist.index, fontsize=9)
        ax2.set_xlabel('Number of Tours', fontweight='bold')
        ax2.set_title('Activity Type Distribution', fontweight='bold', pad=10)
        ax2.grid(axis='x', alpha=0.3)
        
        for i, count in enumerate(act_dist.values):
            pct = 100 * count / len(seg_df)
            ax2.text(count + count*0.02, i, f'{count:,} ({pct:.1f}%)', va='center', fontsize=8, fontweight='bold')
        
        # Chart 3: Feature Count Distribution
        ax3 = axes[0, 2]
        feature_dist = seg_df['num_features'].value_counts().sort_index()
        colors_multi = ['#d62728', '#ff7f0e', '#bcbd22', '#2ca02c', '#1f77b4', '#9467bd']
        bars = ax3.bar(range(len(feature_dist)), feature_dist.values, 
                       color=colors_multi[:len(feature_dist)], alpha=0.8, edgecolor='black', linewidth=1.5)
        ax3.set_xticks(range(len(feature_dist)))
        ax3.set_xticklabels([f'{int(x)}' for x in feature_dist.index])
        ax3.set_ylabel('Number of Tours', fontweight='bold')
        ax3.set_xlabel('Number of Features', fontweight='bold')
        ax3.set_title('Feature Sophistication', fontweight='bold', pad=10)
        ax3.grid(axis='y', alpha=0.3)
        
        for i, count in enumerate(feature_dist.values):
            pct = 100 * count / len(seg_df)
            ax3.text(i, count + count*0.02, f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=8, fontweight='bold')
        
        # Chart 4: Feature Adoption by Activity Type (Heatmap) - spans all 3 columns in row 2
        ax4 = plt.subplot2grid((2, 3), (1, 0), colspan=3, fig=fig)
        seg_activities = seg_df['activity_type'].value_counts().head(8).index.tolist()
        heatmap_data = []
        for activity in seg_activities:
            act_df = seg_df[seg_df['activity_type'] == activity]
            row = []
            for col in PRICING_FEATURES.keys():
                adoption = 100 * act_df[col].sum() / len(act_df) if len(act_df) > 0 else 0
                row.append(adoption)
            heatmap_data.append(row)
        
        heatmap_act_df = pd.DataFrame(heatmap_data, columns=list(PRICING_FEATURES.values()), index=seg_activities)
        sns.heatmap(heatmap_act_df, annot=True, fmt='.1f', cmap='Greens', ax=ax4, 
                    cbar_kws={'label': 'Adoption %'}, linewidths=2, linecolor='white', vmin=0, vmax=70)
        ax4.set_xlabel('Pricing Features', fontweight='bold', fontsize=12)
        ax4.set_ylabel('Activity Type', fontweight='bold', fontsize=12)
        ax4.set_title(f'Feature Adoption by Activity Type (% of tours)', fontweight='bold', fontsize=13, pad=15)
        ax4.set_yticklabels(seg_activities, rotation=0, fontsize=10)
        ax4.set_xticklabels(list(PRICING_FEATURES.values()), rotation=45, ha='right', fontsize=10)
        
        plt.tight_layout()
        plt.show()
    
    print(f"✓ Granular analysis for {tier} complete")


In [0]:
# ============================================================================
# STEP 6: CROSS-TIER COMPARISON SUMMARY
# ============================================================================

print("\n[6/6] Creating cross-tier comparison summary...")

fig = plt.figure(figsize=(24, 14))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3, top=0.95)
fig.suptitle('CROSS-TIER COMPARISON - PRICING FEATURE ADOPTION', fontsize=20, fontweight='bold', y=0.98)

# Chart 1: Feature Adoption Comparison Across Tiers
ax1 = fig.add_subplot(gs[0, :])
comparison_data = []
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    row = [tier]
    for col in PRICING_FEATURES.keys():
        adoption = 100 * tier_df[col].sum() / len(tier_df) if len(tier_df) > 0 else 0
        row.append(adoption)
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data, columns=['Tier'] + list(PRICING_FEATURES.values()))
comparison_df = comparison_df.set_index('Tier')

x = np.arange(len(GMV_TIERS))
width = 0.15
colors_feat = ['#2ca02c', '#ff7f0e', '#9467bd', '#17becf', '#d62728']

for i, (feature, color) in enumerate(zip(PRICING_FEATURES.values(), colors_feat)):
    offset = width * (i - 2)
    bars = ax1.bar(x + offset, comparison_df[feature], width, label=feature, color=color, alpha=0.8, edgecolor='black', linewidth=1.5)
    
    for j, val in enumerate(comparison_df[feature]):
        ax1.text(j + offset, val + 1, f'{val:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold', rotation=90)

ax1.set_xlabel('GMV Tier', fontweight='bold', fontsize=13)
ax1.set_ylabel('Adoption Rate (%)', fontweight='bold', fontsize=13)
ax1.set_title('Feature Adoption Rates Across GMV Tiers', fontweight='bold', fontsize=15, pad=15)
ax1.set_xticks(x)
ax1.set_xticklabels(GMV_TIERS, fontsize=12, fontweight='bold')
ax1.legend(loc='upper left', fontsize=11, framealpha=0.9)
ax1.grid(axis='y', alpha=0.3)

# Chart 2: Average Features per Tour by Tier
ax2 = fig.add_subplot(gs[1, 0])
avg_features = df.groupby('gmv_tier')['num_features'].mean().reindex(GMV_TIERS)
colors_tier = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']
bars = ax2.bar(range(len(avg_features)), avg_features.values, color=colors_tier, alpha=0.8, edgecolor='black', linewidth=2)
ax2.set_xticks(range(len(avg_features)))
ax2.set_xticklabels(GMV_TIERS, fontsize=11, fontweight='bold')
ax2.set_ylabel('Avg Features per Tour', fontweight='bold', fontsize=12)
ax2.set_title('Average Feature Sophistication', fontweight='bold', fontsize=14, pad=10)
ax2.grid(axis='y', alpha=0.3)

for i, val in enumerate(avg_features.values):
    ax2.text(i, val + 0.05, f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Chart 3: Tours with 3+ Features by Tier
ax3 = fig.add_subplot(gs[1, 1])
power_users = []
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    pct = 100 * len(tier_df[tier_df['num_features'] >= 3]) / len(tier_df) if len(tier_df) > 0 else 0
    power_users.append(pct)

bars = ax3.bar(range(len(power_users)), power_users, color=colors_tier, alpha=0.8, edgecolor='black', linewidth=2)
ax3.set_xticks(range(len(GMV_TIERS)))
ax3.set_xticklabels(GMV_TIERS, fontsize=11, fontweight='bold')
ax3.set_ylabel('% of Tours', fontweight='bold', fontsize=12)
ax3.set_title('Power Users (3+ Features)', fontweight='bold', fontsize=14, pad=10)
ax3.grid(axis='y', alpha=0.3)

for i, val in enumerate(power_users):
    ax3.text(i, val + 0.5, f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Chart 4: Tours with 0 Features by Tier
ax4 = fig.add_subplot(gs[1, 2])
non_adopters = []
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    pct = 100 * len(tier_df[tier_df['num_features'] == 0]) / len(tier_df) if len(tier_df) > 0 else 0
    non_adopters.append(pct)

bars = ax4.bar(range(len(non_adopters)), non_adopters, color=colors_tier, alpha=0.8, edgecolor='black', linewidth=2)
ax4.set_xticks(range(len(GMV_TIERS)))
ax4.set_xticklabels(GMV_TIERS, fontsize=11, fontweight='bold')
ax4.set_ylabel('% of Tours', fontweight='bold', fontsize=12)
ax4.set_title('Non-Adopters (0 Features)', fontweight='bold', fontsize=14, pad=10)
ax4.grid(axis='y', alpha=0.3)

for i, val in enumerate(non_adopters):
    ax4.text(i, val + 0.5, f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Chart 5: Summary Table
ax5 = fig.add_subplot(gs[2, :])
ax5.axis('off')

summary_table = []
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    summary_table.append([
        tier,
        f'{len(tier_df):,}',
        f'{tier_df["supplier_id"].nunique():,}',
        f'€{tier_df["gmv_l12mo"].sum()/1e6:.1f}M',
        f'{tier_df["num_features"].mean():.2f}',
        f'{100*len(tier_df[tier_df["num_features"]==0])/len(tier_df):.1f}%',
        f'{100*len(tier_df[tier_df["num_features"]>=3])/len(tier_df):.1f}%',
        f'{100*tier_df["has_group_pricing"].sum()/len(tier_df):.1f}%',
        f'{100*tier_df["has_addons"].sum()/len(tier_df):.1f}%',
        f'{100*tier_df["uses_live_pricing"].sum()/len(tier_df):.1f}%'
    ])

table = ax5.table(
    cellText=summary_table,
    colLabels=['GMV Tier', 'Tours', 'Suppliers', 'Total GMV', 'Avg Features', 
               'Non-Adopters', 'Power Users', 'Group Pricing', 'Add-ons', 'Live Pricing'],
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 3)

for i in range(10):
    table[(0, i)].set_facecolor('#1f77b4')
    table[(0, i)].set_text_props(weight='bold', color='white')

for i in range(1, 5):
    for j in range(10):
        table[(i, j)].set_facecolor(['#ffebee', '#fff3e0', '#e8f5e9', '#e3f2fd'][i-1])

ax5.set_title('COMPREHENSIVE CROSS-TIER SUMMARY', fontweight='bold', fontsize=16, pad=30, y=0.95)

plt.tight_layout()
plt.show()

print("\n✓ Cross-tier comparison complete")

print("\n" + "="*100)
print("PRICING FEATURE ADOPTION AUDIT - COMPLETE")
print("="*100)


In [0]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*100)
print("PRICING FEATURE ADOPTION AUDIT - COMPLETE")
print("="*100)
print(f"\nTotal Tours Analyzed: {len(df):,}")
print(f"Total Suppliers: {df['supplier_id'].nunique():,}")
print(f"Total GMV (L12M): €{df['gmv_l12mo'].sum()/1e6:.1f}M")
print(f"\nGMV Tier Breakdown:")
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    print(f"  {tier}: {len(tier_df):,} tours ({100*len(tier_df)/len(df):.1f}%) | €{tier_df['gmv_l12mo'].sum()/1e6:.1f}M GMV")

print(f"\nOverall Feature Adoption:")
for col, name in PRICING_FEATURES.items():
    adoption = 100 * df[col].sum() / len(df)
    tours = df[col].sum()
    print(f"  {name}: {tours:,} tours ({adoption:.1f}%)")

print(f"\nFeature Sophistication:")
for i in range(6):
    count = len(df[df['num_features'] == i])
    pct = 100 * count / len(df)
    print(f"  {i} features: {count:,} tours ({pct:.1f}%)")

print("\n" + "="*100)
print("All visualizations saved to /dbfs/mnt/data/")
print("="*100)
